# VS_CNN_Baselines_AlarmEval — Batch alarm-aggregated re-evaluation

Re-runs the three PyTorch baselines (`VS_BandPower_MLP`, `VS_Coherence_Matrix`,
`VS_Undirected_Correlation`) with the literature-standard alarm aggregation
applied at the per-fold evaluation step (K=5 of M=12 vote, 30-min refractory).
AUC and AUC-PR are threshold-free and identical to the originals; only
sensitivity, specificity and FPR/h change.

**Expected runtime: 1.5–3 hours** (EDF loading + feature computation +
CNN training × 21 folds × 3 baselines). Run in sequence; intermediate
results saved after each baseline so a crash does not lose earlier work.


In [1]:
# --- portable repo bootstrap (added for public release; locates the repo root) ---
import sys as _sys, pathlib as _pl
REPO = _pl.Path.cwd()
while not (REPO / 'src' / 'config.py').exists() and REPO != REPO.parent:
    REPO = REPO.parent
_sys.path.insert(0, str(REPO / 'src'))
from pathlib import Path
CODE_DIR = str(REPO); CODE = REPO; CODEV2 = REPO; PROJECT_DIR = REPO
# --------------------------------------------------------------------------------

# Cell 0 — Imports
import os, sys, json, copy, time, warnings
from pathlib import Path
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from scipy.signal import welch, coherence

# [path set by bootstrap] CODE_DIR = r"<repo>/Code"
sys.path.insert(0, CODE_DIR)
from config import (
    DATA_ROOT, CANONICAL_CHANNELS, N_CHANNELS, FS, STEP_SEC,
    EXCLUDED_PATIENTS, RESULTS_DIR, RANDOM_SEED,
    ALARM_K, ALARM_M, ALARM_REFRACTORY,
)
from summary_parser import parse_all_summaries
from data_loader   import load_edf
from preprocessing import bandpass_filter, segment_signal, label_windows
from metrics       import evaluate_with_alarms
from sklearn.metrics import roc_curve, roc_auc_score, average_precision_score
from sklearn.model_selection import train_test_split

torch.manual_seed(RANDOM_SEED); np.random.seed(RANDOM_SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}  Alarm K={ALARM_K}/M={ALARM_M}, refractory={ALARM_REFRACTORY*STEP_SEC//60}min')

BANDS = {'delta':(0.5,4),'theta':(4,8),'alpha':(8,13),'beta':(13,30),'broadband':(0.5,30)}
N_BP_FEAT = N_CHANNELS * len(BANDS)

all_seizures = parse_all_summaries(DATA_ROOT)
patients = sorted([p for p in os.listdir(DATA_ROOT)
                   if os.path.isdir(os.path.join(DATA_ROOT, p))
                   and p.startswith('chb') and p not in EXCLUDED_PATIENTS])
print(f'Patients available: {len(patients)}')


Device: cpu  Alarm K=5/M=12, refractory=30min
[INFO] chb01: 7 seizure-containing files, 7 total seizures
[INFO] chb02: 3 seizure-containing files, 3 total seizures
[INFO] chb03: 7 seizure-containing files, 7 total seizures
[INFO] chb04: 3 seizure-containing files, 4 total seizures
[INFO] chb05: 5 seizure-containing files, 5 total seizures
[INFO] chb06: 7 seizure-containing files, 10 total seizures
[INFO] chb07: 3 seizure-containing files, 3 total seizures
[INFO] chb08: 5 seizure-containing files, 5 total seizures
[INFO] chb09: 3 seizure-containing files, 4 total seizures
[INFO] chb10: 7 seizure-containing files, 7 total seizures
[INFO] chb11: 3 seizure-containing files, 3 total seizures
[INFO] chb12: 13 seizure-containing files, 40 total seizures
[INFO] chb13: 8 seizure-containing files, 12 total seizures
[INFO] chb14: 7 seizure-containing files, 8 total seizures
[INFO] chb15: 14 seizure-containing files, 20 total seizures
[INFO] chb16: 6 seizure-containing files, 10 total seizures
[IN

In [2]:
# Cell 1 — Feature extraction functions (from each original baseline)

def band_power_features(window, fs, bands=BANDS):
    freqs, psd = welch(window, fs=fs, nperseg=min(256, window.shape[1]))
    feats = []
    for lo, hi in bands.values():
        mask = (freqs >= lo) & (freqs <= hi)
        bp = np.trapz(psd[:, mask], freqs[mask], axis=1)
        feats.append(np.log1p(bp))
    f = np.concatenate(feats)
    return ((f - f.mean()) / (f.std() + 1e-8)).astype(np.float32)

def coherence_matrix(window, fs):
    n_ch = window.shape[0]; mat = np.eye(n_ch, dtype=np.float32)
    for i in range(n_ch):
        for j in range(i+1, n_ch):
            _, Cxy = coherence(window[i], window[j], fs=fs, nperseg=min(256, window.shape[1]))
            c = float(Cxy.mean()); mat[i, j] = c; mat[j, i] = c
    return mat

def pearson_matrix(window):
    return np.corrcoef(window).astype(np.float32)

print('Feature functions ready.')


Feature functions ready.


In [3]:
# Cell 2 — Build per-patient feature dictionaries (all 3 baselines, in one EDF pass)

def build_features():
    bp, coh, corr = {}, {}, {}
    for pid in patients:
        fmap = all_seizures.get(pid, {})
        if not fmap: continue
        print(f'\nPatient {pid}')
        bp_X, coh_X, corr_X, ys = [], [], [], []
        for fname in sorted(fmap):
            seizures = fmap[fname]
            if not seizures: continue
            edf_path = os.path.join(DATA_ROOT, pid, fname)
            if not os.path.exists(edf_path): continue
            try:
                signal, fs = load_edf(edf_path)
            except Exception:
                continue
            filtered = bandpass_filter(signal)
            windows  = segment_signal(filtered)
            labels   = label_windows(filtered.shape[1], seizures)
            valid = labels != -1
            windows = windows[valid]; labels = labels[valid]
            if (labels == 1).sum() == 0: continue
            bp_X.append(np.stack([band_power_features(w, fs) for w in windows]))
            coh_X.append(np.stack([coherence_matrix(w, fs) for w in windows]))
            corr_X.append(np.stack([pearson_matrix(w) for w in windows]))
            ys.append(labels)
            print(f'  {fname}: pre={(labels==1).sum()} int={(labels==0).sum()}')
        if not ys: continue
        y_all = np.concatenate(ys)
        bp[pid]   = (np.concatenate(bp_X),   y_all)
        coh[pid]  = (np.concatenate(coh_X),  y_all)
        corr[pid] = (np.concatenate(corr_X), y_all)
    return bp, coh, corr

t0 = time.time()
bp_data, coh_data, corr_data = build_features()
print(f'\nFeature extraction done in {(time.time()-t0)/60:.1f}min')
print(f'  Band-power patients: {len(bp_data)}')
print(f'  Coherence patients:  {len(coh_data)}')
print(f'  Correlation patients:{len(corr_data)}')
patient_ids = sorted(bp_data.keys())



Patient chb01
  chb01_03.edf: pre=148 int=146
    [LABEL] Seizure at 1467s: preictal window out of bounds (would start at -333s) — skipping preictal label.
    [LABEL] Seizure at 1732s: preictal window out of bounds (would start at -68s) — skipping preictal label.
    [LABEL] Seizure at 1015s: preictal window out of bounds (would start at -785s) — skipping preictal label.
    [LABEL] Seizure at 1720s: preictal window out of bounds (would start at -80s) — skipping preictal label.
    [LABEL] Seizure at 327s: preictal window out of bounds (would start at -1473s) — skipping preictal label.
  chb01_26.edf: pre=148 int=33

Patient chb02
  chb02_16+.edf: pre=148 int=144
    [LABEL] Seizure at 130s: preictal window out of bounds (would start at -1670s) — skipping preictal label.
  chb02_19.edf: pre=148 int=183

Patient chb03
    [LABEL] Seizure at 362s: preictal window out of bounds (would start at -1438s) — skipping preictal label.
    [LABEL] Seizure at 731s: preictal window out of bounds 

In [4]:
# Cell 3 — Architectures (identical to the originals)

class BandPowerMLP(nn.Module):
    def __init__(self, n_feat=N_BP_FEAT, dropout=0.5):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_feat, 256), nn.BatchNorm1d(256), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(256, 128),    nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(128, 64),     nn.ReLU(),
            nn.Linear(64, 1),       nn.Sigmoid())
    def forward(self, x): return self.net(x).squeeze(1)

class MatrixCNN(nn.Module):
    def __init__(self, n_ch=N_CHANNELS, dropout=0.5):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(True),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(True), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(True), nn.MaxPool2d(2))
        with torch.no_grad():
            flat = self.features(torch.zeros(1,1,n_ch,n_ch)).numel()
        self.classifier = nn.Sequential(
            nn.Flatten(), nn.Dropout(dropout),
            nn.Linear(flat, 256), nn.ReLU(True), nn.Dropout(0.3),
            nn.Linear(256, 1), nn.Sigmoid())
    def forward(self, x): return self.classifier(self.features(x)).squeeze(1)


class TensorDataset2D(Dataset):
    def __init__(self, X, y, kind='vec'):
        if kind == 'vec':
            self.X = torch.tensor(X, dtype=torch.float32)
        else:
            self.X = torch.tensor(X, dtype=torch.float32).unsqueeze(1)
        self.y = torch.tensor(y, dtype=torch.float32)
    def __len__(self): return len(self.y)
    def __getitem__(self, i): return self.X[i], self.y[i]


def train_one(model, X_tr, y_tr, X_va, y_va, kind, epochs=20, patience=5, lr=1e-3, bs=64):
    opt   = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    pos_w = (len(y_tr) - (y_tr == 1).sum()) / max((y_tr == 1).sum(), 1)
    bce   = nn.BCELoss(reduction='none')
    ds_tr = TensorDataset2D(X_tr, y_tr, kind)
    ld_tr = DataLoader(ds_tr, batch_size=bs, shuffle=True, drop_last=True)
    best  = float('inf'); best_state = None; patience_ct = 0
    for ep in range(epochs):
        model.train()
        for xb, yb in ld_tr:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            p = model(xb).clamp(1e-7, 1 - 1e-7)
            w = torch.where(yb == 1, torch.full_like(yb, float(pos_w)), torch.ones_like(yb))
            loss = (w * bce(p, yb)).mean()
            opt.zero_grad(); loss.backward(); opt.step()
        # val
        model.eval()
        with torch.no_grad():
            x_v = torch.tensor(X_va, dtype=torch.float32)
            if kind != 'vec': x_v = x_v.unsqueeze(1)
            x_v = x_v.to(DEVICE); y_v = torch.tensor(y_va, dtype=torch.float32).to(DEVICE)
            p_v = model(x_v).clamp(1e-7, 1 - 1e-7)
            vl = (bce(p_v, y_v)).mean().item()
        if vl < best:
            best = vl; best_state = copy.deepcopy(model.state_dict()); patience_ct = 0
        else:
            patience_ct += 1
            if patience_ct >= patience: break
    if best_state is not None:
        model.load_state_dict(best_state)
    return model

@torch.no_grad()
def predict(model, X, kind):
    model.eval()
    x = torch.tensor(X, dtype=torch.float32)
    if kind != 'vec': x = x.unsqueeze(1)
    return model(x.to(DEVICE)).cpu().numpy()

print('Architectures and trainer ready.')


Architectures and trainer ready.


In [5]:
# Cell 4 — Generic LOPO + alarm pipeline (smoothing + alarm-level operational threshold)
# Post-processing follows Truong 2018 / Bandarabadi 2015 / Mormann 2007:
#   raw probs → moving-average smoothing
#             → operational threshold (Sens maximised s.t. ALARM-level FPR/h ≤ 1.0)
#             → K-of-M sliding-window vote → 30-min refractory suppression.
from alarm_postproc import smooth_probs, operational_threshold

SMOOTH_WINDOW = 10           # 10 windows ≈ 100 s (matches M=12 voting window)
TARGET_FPR_H  = 1.0          # Mormann 2007 clinical threshold (alarm-level)

def run_lopo_baseline(data, model_factory, kind, baseline_name, max_train=None):
    rows_alarm, rows_cmp = [], []
    pids = sorted(data.keys())
    print(f'\n== {baseline_name} ({len(pids)} folds, smooth={SMOOTH_WINDOW}w, alarm-level FPR/h≤{TARGET_FPR_H}) ==')
    t0 = time.time()
    for fi, test_pid in enumerate(pids, 1):
        Xs = [data[p][0] for p in pids if p != test_pid]
        ys = [data[p][1] for p in pids if p != test_pid]
        X_tr = np.concatenate(Xs); y_tr = np.concatenate(ys)
        X_te, y_te = data[test_pid]
        if max_train and len(X_tr) > max_train:
            rng = np.random.default_rng(RANDOM_SEED + fi)
            idx = rng.choice(len(X_tr), size=max_train, replace=False)
            X_tr, y_tr = X_tr[idx], y_tr[idx]
        tr_i, va_i = train_test_split(np.arange(len(y_tr)), test_size=0.15,
                                      random_state=RANDOM_SEED, stratify=y_tr)
        X_va, y_va = X_tr[va_i], y_tr[va_i]
        X_tr, y_tr = X_tr[tr_i], y_tr[tr_i]
        model = model_factory().to(DEVICE)
        model = train_one(model, X_tr, y_tr, X_va, y_va, kind)
        probs_raw = predict(model, X_te, kind)
        if len(np.unique(y_te)) < 2: continue

        # --- post-processing pipeline ---
        probs    = smooth_probs(probs_raw, window=SMOOTH_WINDOW)
        threshold = operational_threshold(
            probs, y_te, STEP_SEC, target_fpr_h=TARGET_FPR_H,
            K=ALARM_K, M=ALARM_M, refractory_windows=ALARM_REFRACTORY,
        )

        a = evaluate_with_alarms(
            y_true=y_te, probs=probs, threshold=threshold,
            K=ALARM_K, M=ALARM_M, refractory_windows=ALARM_REFRACTORY,
            patient_id=test_pid,
        )
        a['model'] = baseline_name; a['threshold'] = threshold
        rows_alarm.append(a)

        # window-level reference (raw probs, raw Youden-J)
        fpr_a, tpr_a, thr = roc_curve(y_te, probs_raw)
        thr_raw = float(np.clip(thr[np.argmax(tpr_a - fpr_a)], 0.05, 0.95))
        pred_w = (probs_raw >= thr_raw).astype(int)
        tp=int(((pred_w==1)&(y_te==1)).sum()); fp=int(((pred_w==1)&(y_te==0)).sum())
        tn=int(((pred_w==0)&(y_te==0)).sum()); fn=int(((pred_w==0)&(y_te==1)).sum())
        sens_w = tp/max(tp+fn,1)
        hours_int = (y_te==0).sum() * STEP_SEC / 3600.0
        fpr_w = fp/max(hours_int, 1e-9)
        rows_cmp.append(dict(patient=test_pid, model=baseline_name,
            auc=a['auc'], auc_pr=a['auc_pr'], threshold=threshold,
            sens_window=sens_w, fpr_h_window=fpr_w,
            sens_alarm=a['sensitivity'], fpr_h_alarm=a['fpr_per_hour']))
        print(f'  [{fi:2d}/{len(pids)}] {test_pid}  AUC={a["auc"]:.3f}  '
              f'thr={threshold:.2f}  Sens(a)={a["sensitivity"]:.3f}  '
              f'FPR/h(w)={fpr_w:.1f} → FPR/h(a)={a["fpr_per_hour"]:.2f}')
    print(f'  {baseline_name} done in {(time.time()-t0)/60:.1f}min')
    return rows_alarm, rows_cmp


In [6]:
# Cell 5 — Run Band Power MLP
bp_alarm, bp_cmp = run_lopo_baseline(bp_data, lambda: BandPowerMLP(N_BP_FEAT),
                                      kind='vec', baseline_name='BandPower_MLP')
pd.DataFrame(bp_alarm).to_csv(Path(RESULTS_DIR) / 'lopo_bp_alarm.csv', index=False)
pd.DataFrame(bp_cmp).to_csv(Path(RESULTS_DIR) / 'lopo_bp_compare.csv', index=False)



== BandPower_MLP (21 folds, smooth=10w, alarm-level FPR/h≤1.0) ==
  [ 1/21] chb01  AUC=0.364  thr=0.69  Sens(a)=0.000  FPR/h(w)=4.0 → FPR/h(a)=0.00
  [ 2/21] chb02  AUC=0.434  thr=0.65  Sens(a)=0.003  FPR/h(w)=49.5 → FPR/h(a)=0.00
  [ 3/21] chb03  AUC=0.560  thr=0.59  Sens(a)=0.000  FPR/h(w)=53.8 → FPR/h(a)=0.00
  [ 4/21] chb04  AUC=0.466  thr=0.64  Sens(a)=0.007  FPR/h(w)=19.9 → FPR/h(a)=0.84
  [ 5/21] chb05  AUC=0.606  thr=0.42  Sens(a)=0.007  FPR/h(w)=83.2 → FPR/h(a)=0.00
  [ 6/21] chb06  AUC=0.517  thr=0.75  Sens(a)=0.001  FPR/h(w)=267.1 → FPR/h(a)=1.00
  [ 7/21] chb07  AUC=0.479  thr=0.64  Sens(a)=0.002  FPR/h(w)=352.3 → FPR/h(a)=0.87
  [ 8/21] chb08  AUC=0.438  thr=0.66  Sens(a)=0.000  FPR/h(w)=292.7 → FPR/h(a)=0.67
  [ 9/21] chb09  AUC=0.451  thr=0.75  Sens(a)=0.002  FPR/h(w)=314.0 → FPR/h(a)=0.97
  [10/21] chb10  AUC=0.551  thr=0.70  Sens(a)=0.002  FPR/h(w)=176.0 → FPR/h(a)=1.00
  [11/21] chb13  AUC=0.411  thr=0.72  Sens(a)=0.000  FPR/h(w)=4.4 → FPR/h(a)=0.00
  [12/21] chb14  

In [7]:
# Cell 6 — Run Coherence CNN
coh_alarm, coh_cmp = run_lopo_baseline(coh_data, MatrixCNN,
                                        kind='matrix', baseline_name='Coherence_CNN')
pd.DataFrame(coh_alarm).to_csv(Path(RESULTS_DIR) / 'lopo_coh_alarm.csv', index=False)
pd.DataFrame(coh_cmp).to_csv(Path(RESULTS_DIR) / 'lopo_coh_compare.csv', index=False)



== Coherence_CNN (21 folds, smooth=10w, alarm-level FPR/h≤1.0) ==
  [ 1/21] chb01  AUC=0.291  thr=0.85  Sens(a)=0.000  FPR/h(w)=16.1 → FPR/h(a)=0.00
  [ 2/21] chb02  AUC=0.366  thr=0.75  Sens(a)=0.000  FPR/h(w)=1.1 → FPR/h(a)=0.00
  [ 3/21] chb03  AUC=0.561  thr=0.84  Sens(a)=0.002  FPR/h(w)=200.2 → FPR/h(a)=0.00
  [ 4/21] chb04  AUC=0.375  thr=0.56  Sens(a)=0.000  FPR/h(w)=291.8 → FPR/h(a)=0.84
  [ 5/21] chb05  AUC=0.502  thr=0.68  Sens(a)=0.002  FPR/h(w)=73.1 → FPR/h(a)=0.00
  [ 6/21] chb06  AUC=0.521  thr=0.62  Sens(a)=0.003  FPR/h(w)=262.8 → FPR/h(a)=1.00
  [ 7/21] chb07  AUC=0.177  thr=0.75  Sens(a)=0.000  FPR/h(w)=343.2 → FPR/h(a)=0.72
  [ 8/21] chb08  AUC=0.619  thr=0.69  Sens(a)=0.007  FPR/h(w)=233.5 → FPR/h(a)=0.67
  [ 9/21] chb09  AUC=0.650  thr=0.59  Sens(a)=0.005  FPR/h(w)=246.6 → FPR/h(a)=0.97
  [10/21] chb10  AUC=0.511  thr=0.53  Sens(a)=0.002  FPR/h(w)=264.7 → FPR/h(a)=1.00
  [11/21] chb13  AUC=0.630  thr=0.71  Sens(a)=0.007  FPR/h(w)=239.3 → FPR/h(a)=0.00
  [12/21] chb

In [8]:
# Cell 7 — Run Undirected Correlation CNN
corr_alarm, corr_cmp = run_lopo_baseline(corr_data, MatrixCNN,
                                          kind='matrix', baseline_name='Correlation_CNN')
pd.DataFrame(corr_alarm).to_csv(Path(RESULTS_DIR) / 'lopo_corr_alarm.csv', index=False)
pd.DataFrame(corr_cmp).to_csv(Path(RESULTS_DIR) / 'lopo_corr_compare.csv', index=False)



== Correlation_CNN (21 folds, smooth=10w, alarm-level FPR/h≤1.0) ==
  [ 1/21] chb01  AUC=0.399  thr=0.49  Sens(a)=0.003  FPR/h(w)=215.2 → FPR/h(a)=0.00
  [ 2/21] chb02  AUC=0.515  thr=0.77  Sens(a)=0.003  FPR/h(w)=48.4 → FPR/h(a)=0.00
  [ 3/21] chb03  AUC=0.630  thr=0.84  Sens(a)=0.002  FPR/h(w)=210.3 → FPR/h(a)=0.00
  [ 4/21] chb04  AUC=0.353  thr=0.70  Sens(a)=0.000  FPR/h(w)=6.3 → FPR/h(a)=0.84
  [ 5/21] chb05  AUC=0.449  thr=0.06  Sens(a)=0.007  FPR/h(w)=1.4 → FPR/h(a)=0.00
  [ 6/21] chb06  AUC=0.352  thr=0.58  Sens(a)=0.002  FPR/h(w)=6.5 → FPR/h(a)=0.93
  [ 7/21] chb07  AUC=0.495  thr=0.62  Sens(a)=0.002  FPR/h(w)=125.6 → FPR/h(a)=0.87
  [ 8/21] chb08  AUC=0.501  thr=0.60  Sens(a)=0.004  FPR/h(w)=222.1 → FPR/h(a)=0.67
  [ 9/21] chb09  AUC=0.402  thr=0.82  Sens(a)=0.003  FPR/h(w)=318.9 → FPR/h(a)=0.97
  [10/21] chb10  AUC=0.551  thr=0.50  Sens(a)=0.006  FPR/h(w)=291.8 → FPR/h(a)=1.00
  [11/21] chb13  AUC=0.473  thr=0.94  Sens(a)=0.000  FPR/h(w)=74.6 → FPR/h(a)=0.00
  [12/21] chb14

In [9]:
# Cell 8 — Final summary
print(f'{"Baseline":<18} {"AUC":>6} {"AUC-PR":>7} '
      f'{"Sens(w)":>8} {"FPR/h(w)":>10} {"Sens(a)":>8} {"FPR/h(a)":>10}')
print('-' * 75)
for name, df in [('BandPower_MLP', pd.DataFrame(bp_cmp)),
                 ('Coherence_CNN', pd.DataFrame(coh_cmp)),
                 ('Correlation_CNN', pd.DataFrame(corr_cmp))]:
    if df.empty: continue
    print(f'{name:<18} {df.auc.mean():>6.3f} {df.auc_pr.mean():>7.3f} '
          f'{df.sens_window.mean():>8.3f} {df.fpr_h_window.mean():>10.1f} '
          f'{df.sens_alarm.mean():>8.3f} {df.fpr_h_alarm.mean():>10.2f}')

print('\nLiterature reference: Truong 2018 ≈ 0.16/h | Khan 2018 ≈ 0.14/h | clinical ~1/h')
print('Outputs: lopo_{bp,coh,corr}_{alarm,compare}.csv')


Baseline              AUC  AUC-PR  Sens(w)   FPR/h(w)  Sens(a)   FPR/h(a)
---------------------------------------------------------------------------
BandPower_MLP       0.515   0.500    0.486      136.9    0.002       0.35
Coherence_CNN       0.511   0.502    0.541      171.9    0.003       0.38
Correlation_CNN     0.497   0.490    0.503      161.4    0.004       0.36

Literature reference: Truong 2018 ≈ 0.16/h | Khan 2018 ≈ 0.14/h | clinical ~1/h
Outputs: lopo_{bp,coh,corr}_{alarm,compare}.csv
